#ChEMBL Pathway Extraction

Purpose: extract pathway related to a specific gene


Output:
gene_pathways.csv containing:

In [3]:
#import packages
import requests
import pandas as pd

In [4]:
#Before pipeline: test connecting and extracting pathway for one gene

gene = "PTGS2"
url = f"https://reactome.org/ContentService/data/mapping/UniProt/P35354/pathways"

response = requests.get(url)
response.json()


#Question: "What unique proteins do I need to query Reactome for?"
#answer/output: ['P35354', 'P23219', 'P35367', 'P12821', 'P08588', 'Q13936', 'Q13698', 'Q01668', 'O60840', 'P31645', 'P06213']




[{'dbId': 2142770,
  'displayName': 'Synthesis of 15-eicosatetraenoic acid derivatives',
  'stId': 'R-HSA-2142770',
  'stIdVersion': 'R-HSA-2142770.2',
  'isInDisease': False,
  'isInferred': False,
  'maxDepth': 2,
  'name': ['Synthesis of 15-eicosatetraenoic acid derivatives'],
  'releaseDate': '2012-12-04',
  'speciesName': 'Homo sapiens',
  'hasDiagram': False,
  'hasEHLD': False,
  'schemaClass': 'Pathway',
  'className': 'Pathway'},
 {'dbId': 2162123,
  'displayName': 'Synthesis of Prostaglandins (PG) and Thromboxanes (TX)',
  'stId': 'R-HSA-2162123',
  'stIdVersion': 'R-HSA-2162123.6',
  'isInDisease': False,
  'isInferred': False,
  'maxDepth': 2,
  'name': ['Synthesis of Prostaglandins (PG) and Thromboxanes (TX)'],
  'releaseDate': '2012-12-04',
  'speciesName': 'Homo sapiens',
  'hasDiagram': True,
  'hasEHLD': False,
  'lastUpdatedDate': '2022-06-09',
  'schemaClass': 'Pathway',
  'className': 'Pathway'},
 {'dbId': 6783783,
  'displayName': 'Interleukin-10 signaling',
  'stI

In [11]:
#pathway extraction function
def get_pathway(uniprot_id):
    url = f"https://reactome.org/ContentService/data/mapping/UniProt/{uniprot_id}/pathways"

    response = requests.get(url)
    pathways = response.json()

    results = []

    #some UniProt proteins do not have a reactome pathway -> skip them
    #correct returns:<class 'list'>
    #wrong returns: <class, 'dict'>

    if isinstance(pathways,list):

        for pathway in pathways:
            
            results.append({
                "UniProt": uniprot_id,
                "Pathway_ID": pathway["stId"],
                "Pathway_Name": pathway["displayName"],
                "Pathway_Status": "Found"
            })
            
            
    #tester to see why crashing
    #print("UniProt", uniprot_id)
    #print(pathways)

    #check response type
    #print(type(pathways))

    else:

        print(f"No pathways found for {uniprot_id}")
            #return []
        results.append({
            "UniProt": uniprot_id,
            "Pathway_ID": None,
            "Pathway_Name": None,
            "Pathway_Status": "Not found"
        })
        

    

    return results

#test
#get_pathway("P35354")


In [12]:
#pipeline -> quries unique proteins

#Question: "What unique proteins do I need to query Reactome for?"
targets = pd.read_csv("C:/Katieryb/Pipelines/proteins/drug_targets.csv")

targets["UniProt"].unique()
#answer/output: ['P35354', 'P23219', 'P35367', 'P12821', 'P08588', 'Q13936', 'Q13698', 'Q01668', 'O60840', 'P31645', 'P06213']


all_pathways = []

for uniprot in targets["UniProt"].unique():

    #add a test for if the uniprot is not valid

    #test marker
    print("Processing:", uniprot)

    pathways = get_pathway(uniprot)

    for pathway in pathways:
        all_pathways.append(pathway)

#df = data frame
#convert to data frame
#pathway_df = pd.DataFrame(all_pathways)
results = pd.DataFrame(all_pathways)
#pathway_df

#save file
#pathway_df.to_csv("C:/Katieryb/Pipelines/proteins/gene_pathways.csv.txt", index = False)
results.to_csv("C:/Katieryb/Pipelines/proteins/gene_pathways.csv.txt", index = False)

#if run from here all takes about -> 3-5 minutes (17 drugs)

Processing: P35354
Processing: P23219
Processing: P35367
Processing: P12821
Processing: P08588
Processing: Q13936
Processing: Q13698
Processing: Q01668
Processing: O60840
No pathways found for O60840
Processing: P31645
Processing: P06213
